# Inspection of FlyWire clustering sweep results

This notebook visualizes the Lightning L4 sweep over:

- **PCA + $k$-means**
- **low-rank vSBM** (single-hop)
- **GNN-vSBM** grid over layers $L$, embedding dim $d$, and learning rate

Primary quality metric: Hungarian alignment against annotated visual neuron types.

```bash
uv run jupyter notebook inspect_sweep_results.ipynb
```

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.optimize import linear_sum_assignment

from evaluate_clustering import (
    align_assignments,
    confusion_matrix,
    evaluate_pair,
    hungarian_score,
    load_assignment_dict,
    load_ground_truth,
    random_baselines,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
PALETTE = {"gnn": "#2a6f97", "lv": "#e76f51", "pca": "#6c757d"}

## 1. Load sweep summary

In [ ]:
results = pd.read_csv("sweep_results.csv")
best = json.loads(Path("sweep_best.json").read_text())


def parse_gnn_name(name: str) -> pd.Series:
    m = re.match(r"gnn_L(\d+)_d(\d+)_lr([0-9.]+)_ent([0-9.]+)", name)
    if not m:
        return pd.Series(
            {"family": name, "layers": np.nan, "d": np.nan, "lr": np.nan, "entropy": np.nan}
        )
    return pd.Series(
        {
            "family": "gnn",
            "layers": int(m.group(1)),
            "d": int(m.group(2)),
            "lr": float(m.group(3)),
            "entropy": float(m.group(4)),
        }
    )


meta = results["name"].apply(parse_gnn_name)
df = pd.concat([results, meta], axis=1)
df.loc[df["name"] == "pca_kmeans", "family"] = "pca"
df.loc[df["name"] == "lv_vsbm", "family"] = "lv"
df = df.sort_values("hungarian", ascending=False).reset_index(drop=True)
df

In [ ]:
print("Best config from sweep_best.json:")
print(json.dumps(best, indent=2))
display(df[["name", "hungarian", "ari", "nmi", "n_pred_clusters"]].head(10))

## 2. Leaderboard

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = df["family"].map(PALETTE).fillna("#333333")
ax.barh(df["name"][::-1], df["hungarian"][::-1], color=colors[::-1])
ax.set_xlabel("Hungarian score vs visual neuron types (higher is better)")
ax.set_title("Sweep leaderboard")
lv_score = float(df.loc[df["family"] == "lv", "hungarian"].iloc[0])
pca_score = float(df.loc[df["family"] == "pca", "hungarian"].iloc[0])
ax.axvline(lv_score, color=PALETTE["lv"], ls="--", lw=1, label="LV baseline")
ax.axvline(pca_score, color=PALETTE["pca"], ls=":", lw=1, label="PCA baseline")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. GNN hyperparameter effects

Heatmaps of Hungarian score over $(L, d)$ for each learning rate.

In [ ]:
gnn = df[df["family"] == "gnn"].copy()
lrs = sorted(gnn["lr"].unique())
fig, axes = plt.subplots(1, len(lrs), figsize=(4.2 * len(lrs), 3.8), sharey=True)
if len(lrs) == 1:
    axes = [axes]

for ax, lr in zip(axes, lrs, strict=True):
    sub = gnn[gnn["lr"] == lr]
    pivot = sub.pivot(index="layers", columns="d", values="hungarian").sort_index()
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="viridis", ax=ax, cbar=(ax is axes[-1]))
    ax.set_title(f"lr = {lr}")
    ax.set_xlabel("d")
    ax.set_ylabel("layers L" if ax is axes[0] else "")

fig.suptitle("GNN-vSBM Hungarian score by hyperparameters", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

sns.lineplot(data=gnn, x="layers", y="hungarian", hue="lr", marker="o", ax=axes[0])
axes[0].set_title("Depth vs score")
axes[0].set_xticks(sorted(gnn["layers"].dropna().unique()))

sns.boxplot(data=gnn, x="d", y="hungarian", ax=axes[1], color=PALETTE["gnn"])
axes[1].set_title("Width $d$ vs score")

sns.boxplot(data=gnn, x="lr", y="hungarian", ax=axes[2], color=PALETTE["gnn"])
axes[2].set_title("Learning rate vs score")

plt.tight_layout()
plt.show()

## 4. Cluster occupancy

How many of the $K=729$ clusters are actually used?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=df,
    x="n_pred_clusters",
    y="hungarian",
    hue="family",
    style="family",
    s=80,
    palette=PALETTE,
    ax=ax,
)
for _, row in df.head(3).iterrows():
    ax.annotate(
        row["name"],
        (row["n_pred_clusters"], row["hungarian"]),
        fontsize=8,
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set_xlabel("Number of nonempty predicted clusters")
ax.set_ylabel("Hungarian score")
ax.set_title("Score vs cluster utilization")
plt.tight_layout()
plt.show()

## 5. Compare selected assignments to ground truth

In [ ]:
gt = load_ground_truth(Path("root_id_type_dict.pkl"))

candidates = {
    "best_gnn": best["pred_path"],
    "lv_sweep": "lv_sweep_assignment_dict.npy",
    "pca_sweep": "pca_sweep_assignment_dict.npy",
}
if not Path(candidates["best_gnn"]).exists():
    candidates["best_gnn"] = "gnn_L1_d64_lr0.005_ent1.0_assignment_dict_729.npy"

preds = {}
for label, path in candidates.items():
    p = Path(path)
    if p.exists():
        preds[label] = load_assignment_dict(p)
        print(f"loaded {label}: {path}")
    else:
        print(f"MISSING {label}: {path}")

In [ ]:
rows = []
gt_labels_ref = None
payloads = {}
for label, pred in preds.items():
    metrics = evaluate_pair(pred, gt)
    gt_labels, pred_labels, shared = align_assignments(pred, gt)
    if gt_labels_ref is None:
        gt_labels_ref = gt_labels
    rows.append({"model": label, **metrics})
    payloads[label] = {
        "dict": pred,
        "labels": pred_labels,
        "gt_labels": gt_labels,
        "shared": shared,
    }

baselines = random_baselines(gt_labels_ref, seed=0)
summary = pd.DataFrame(rows).sort_values("hungarian", ascending=False)
print("Random baselines:", baselines)
display(summary)

### Aligned confusion matrices (top $40\times 40$ block)

Rows = ground-truth types, columns = predicted clusters after Hungarian alignment.

In [ ]:
def aligned_confusion(gt_labels: np.ndarray, pred_labels: np.ndarray) -> np.ndarray:
    conf = confusion_matrix(gt_labels, pred_labels)
    _row_ind, col_ind = linear_sum_assignment(conf, maximize=True)
    aligned = conf[:, col_ind]
    order = np.argsort(-np.diag(aligned))
    return aligned[order][:, order]


n_show = 40
fig, axes = plt.subplots(1, len(payloads), figsize=(4.5 * len(payloads), 4))
if len(payloads) == 1:
    axes = [axes]

for ax, (label, payload) in zip(axes, payloads.items(), strict=True):
    aligned = aligned_confusion(payload["gt_labels"], payload["labels"])
    block = aligned[:n_show, :n_show]
    sns.heatmap(block, ax=ax, cmap="mako", cbar=(ax is axes[-1]))
    score = hungarian_score(payload["gt_labels"], payload["labels"])
    ax.set_title(f"{label}\nHungarian={score:.0f}")
    ax.set_xlabel("predicted (aligned)")
    ax.set_ylabel("ground truth" if ax is axes[0] else "")

plt.tight_layout()
plt.show()

### Cluster size distributions

In [ ]:
fig, axes = plt.subplots(1, len(payloads), figsize=(4.2 * len(payloads), 3.5), sharey=True)
if len(payloads) == 1:
    axes = [axes]

for ax, (label, payload) in zip(axes, payloads.items(), strict=True):
    _ids, counts = np.unique(payload["labels"], return_counts=True)
    ax.hist(counts, bins=40, color=PALETTE["gnn"], alpha=0.85)
    ax.set_title(f"{label}\nmedian size={np.median(counts):.0f}")
    ax.set_xlabel("cluster size")
    ax.set_ylabel("count" if ax is axes[0] else "")

plt.tight_layout()
plt.show()

## 6. Pairwise agreement among models

In [ ]:
labels_map = {k: v["labels"] for k, v in payloads.items()}
keys = list(labels_map)
agree = pd.DataFrame(index=keys, columns=keys, dtype=float)
for a in keys:
    for b in keys:
        agree.loc[a, b] = hungarian_score(labels_map[a], labels_map[b])

display(agree)
fig, ax = plt.subplots(figsize=(4.5, 3.8))
sns.heatmap(agree.astype(float), annot=True, fmt=".0f", cmap="crest", ax=ax)
ax.set_title("Pairwise Hungarian agreement")
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
top = df.iloc[0]
lv = df[df["family"] == "lv"].iloc[0]
pca = df[df["family"] == "pca"].iloc[0]
print(
    f"Best sweep entry: {top['name']} with Hungarian={top['hungarian']:.0f} "
    f"(ARI={top['ari']:.4f}, NMI={top['nmi']:.4f}, nonempty clusters={top['n_pred_clusters']:.0f})."
)
print(f"LV baseline: {lv['hungarian']:.0f}; PCA baseline: {pca['hungarian']:.0f}.")
delta = top["hungarian"] - lv["hungarian"]
verb = "beats" if delta > 0 else "does not beat"
print(f"Best GNN {verb} the matched-budget LV-SBM by {delta:.0f} Hungarian points.")